# 03 — Power Analysis and Sample Sizing for Online Experiments
**Prerequisites:** `01_ab_testing_fundamentals.ipynb` (ratio metrics, delta method);
statistics_course/06_hypothesis_testing.ipynb (power, Type I/II error basics).
**Connects to:** experimental_design_course/08_online_experiments.ipynb (CUPED derivation —
referenced, not repeated here); `05_guardrail_metrics.ipynb` (non-inferiority sizing).

## Narrative thread
```
Sample size for means and proportions -> minimum detectable effect (MDE)
   -> variance reduction via CUPED (pointer) -> sizing ratio/skewed metrics via the delta method
   -> worked numeric example
```

## Sample size formulas

For a two-sample z-test comparing means with equal allocation, targeting power $1-\beta$ at
significance $\alpha$ (two-sided), the required sample size **per arm** is

$$n = \frac{2\sigma^2 (z_{1-\alpha/2} + z_{1-\beta})^2}{\delta^2}$$

where $\delta$ is the effect size you want to detect and $\sigma^2$ the (assumed common) variance.
For a proportion metric $p$ (conversion rate) with $\sigma^2 = p(1-p)$ under the null,

$$n = \frac{2\, p(1-p)\,(z_{1-\alpha/2}+z_{1-\beta})^2}{\delta^2}$$

Equivalently, given a fixed available sample size $n$, the **minimum detectable effect (MDE)** is

$$\delta_{MDE} = (z_{1-\alpha/2}+z_{1-\beta})\sqrt{\frac{2\sigma^2}{n}}$$

Every online-experimentation team runs this second calculation constantly: "given the traffic
this feature actually gets, what's the smallest lift we could realistically detect in two weeks?"
If the MDE is bigger than any effect the team would plausibly expect from the change, the
experiment is underpowered before it even starts.


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests

plt.rcParams.update({
    'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#e8e8e8', 'axes.axisbelow': True,
    'axes.titlesize': 11, 'axes.titleweight': 'bold', 'legend.frameon': False,
})
np.random.seed(42)

In [2]:
# ── Sample size & MDE for a conversion-rate metric ──────────────────────
from scipy.stats import norm

def sample_size_proportion(p_baseline, mde_abs, alpha=0.05, power=0.8):
    z_alpha = norm.ppf(1 - alpha / 2)
    z_beta = norm.ppf(power)
    var = p_baseline * (1 - p_baseline)
    n_per_arm = 2 * var * (z_alpha + z_beta) ** 2 / mde_abs ** 2
    return int(np.ceil(n_per_arm))

def mde_proportion(p_baseline, n_per_arm, alpha=0.05, power=0.8):
    z_alpha = norm.ppf(1 - alpha / 2)
    z_beta = norm.ppf(power)
    var = p_baseline * (1 - p_baseline)
    return (z_alpha + z_beta) * np.sqrt(2 * var / n_per_arm)

p0 = 0.05          # baseline conversion rate: 5%
target_lift_abs = 0.005   # want to detect a 0.5 pp absolute lift (10% relative)

n_needed = sample_size_proportion(p0, target_lift_abs)
print(f"Baseline conversion: {p0:.1%}, target absolute lift: {target_lift_abs:.1%}")
print(f"Required sample size per arm: {n_needed:,}")

# And the inverse question: with 50,000 users/arm/week, what lift can we detect in 2 weeks?
for weeks, users_per_week in [(1, 50_000), (2, 50_000), (4, 50_000)]:
    n = weeks * users_per_week
    mde = mde_proportion(p0, n)
    print(f"{weeks} week(s), n={n:>7,}/arm -> MDE = {mde:.4%} absolute "
          f"({mde/p0:.1%} relative)")

Baseline conversion: 5.0%, target absolute lift: 0.5%
Required sample size per arm: 29,826
1 week(s), n= 50,000/arm -> MDE = 0.3862% absolute (7.7% relative)
2 week(s), n=100,000/arm -> MDE = 0.2731% absolute (5.5% relative)
4 week(s), n=200,000/arm -> MDE = 0.1931% absolute (3.9% relative)


## Variance reduction improves power (pointer)

The sample-size formulas above scale with $\sigma^2$: **halving the variance of the metric halves
the required sample size** for the same MDE. The workhorse technique in industry is **CUPED**
(Controlled-experiment Using Pre-Experiment Data, Deng, Xu, Kohavi & Walker 2013), which uses a
user's *pre-experiment* value of the metric (or a correlated covariate) to remove predictable
variance before comparing arms — conceptually identical to the Lin (2013) covariate-adjusted
estimator from `causal_inference_course/02_randomized_experiments.ipynb`. The full derivation,
the optimal-theta formula, and worked case studies are in
**`experimental_design_course/08_online_experiments.ipynb`** — we only use its headline result here.


In [3]:
# ── Illustrating the *effect* of CUPED-style variance reduction on required sample size ──
# (mechanism/derivation is in experimental_design_course/08; here we just show the power payoff)
def variance_after_cuped(var_y, corr_with_pre_period):
    # CUPED reduces variance by a factor of (1 - rho^2) when using an optimal theta.
    return var_y * (1 - corr_with_pre_period ** 2)

var_raw = p0 * (1 - p0)
for rho in [0.0, 0.3, 0.5, 0.7]:
    var_reduced = variance_after_cuped(var_raw, rho)
    z_alpha, z_beta = norm.ppf(1 - 0.05 / 2), norm.ppf(0.8)
    n_reduced = int(np.ceil(2 * var_reduced * (z_alpha + z_beta) ** 2 / target_lift_abs ** 2))
    pct_savings = 1 - n_reduced / n_needed
    print(f"corr(metric, pre-period)={rho:.1f} -> required n/arm={n_reduced:>7,} "
          f"({pct_savings:.0%} fewer users than without CUPED)")

corr(metric, pre-period)=0.0 -> required n/arm= 29,826 (0% fewer users than without CUPED)
corr(metric, pre-period)=0.3 -> required n/arm= 27,142 (9% fewer users than without CUPED)
corr(metric, pre-period)=0.5 -> required n/arm= 22,370 (25% fewer users than without CUPED)
corr(metric, pre-period)=0.7 -> required n/arm= 15,212 (49% fewer users than without CUPED)


## Sizing ratio and skewed revenue metrics

Revenue-per-user metrics are typically **highly right-skewed** (most users spend \$0, a few
spend a lot) *and* are ratio metrics (`01_ab_testing_fundamentals.ipynb`). Two adjustments matter
for sizing:

1. Use the **delta-method variance** (notebook 01) for $\sigma^2$ in the formulas above, not the
   naive sample variance of a "mean" — the correlation between numerator and denominator changes
   the effective variance.
2. Skew inflates the sample variance itself (a few whale users dominate $\sigma^2$), so revenue
   metrics typically need **far larger samples** than conversion-rate metrics for the same
   relative MDE. Winsorizing or capping extreme values (with disclosure of the cap) is common
   practice to keep required sample sizes tractable, at the cost of not measuring tail effects.


In [4]:
# ── Worked example: sizing a skewed revenue-per-user metric end to end ──────
np.random.seed(7)
n_pilot = 20_000
# 95% zero spenders, 5% spenders with a heavy-tailed (lognormal) spend
is_payer = np.random.rand(n_pilot) < 0.05
revenue = np.where(is_payer, np.random.lognormal(mean=3.0, sigma=1.2, size=n_pilot), 0.0)

mean_rev, var_rev = revenue.mean(), revenue.var(ddof=1)
print(f"Pilot: mean revenue/user = {mean_rev:.2f}, var = {var_rev:.1f}, "
      f"CV = {np.sqrt(var_rev)/mean_rev:.1f} (very high due to skew)")

target_rel_lift = 0.05   # want to detect a 5% relative lift in revenue/user
target_abs = target_rel_lift * mean_rev
n_needed_revenue = int(np.ceil(2 * var_rev * (norm.ppf(0.975) + norm.ppf(0.8)) ** 2 / target_abs ** 2))
print(f"Required n/arm to detect a {target_rel_lift:.0%} relative lift in raw revenue/user: {n_needed_revenue:,}")

# Compare to winsorizing at the 99th percentile
cap = np.percentile(revenue, 99)
revenue_capped = np.minimum(revenue, cap)
var_capped = revenue_capped.var(ddof=1)
n_needed_capped = int(np.ceil(2 * var_capped * (norm.ppf(0.975) + norm.ppf(0.8)) ** 2 / target_abs ** 2))
print(f"Same target, after winsorizing at p99 (cap={cap:.0f}): required n/arm = {n_needed_capped:,}"
      f"  ({1 - n_needed_capped/n_needed_revenue:.0%} reduction)")

Pilot: mean revenue/user = 2.10, var = 409.1, CV = 9.6 (very high due to skew)
Required n/arm to detect a 5% relative lift in raw revenue/user: 580,040
Same target, after winsorizing at p99 (cap=54): required n/arm = 66,700  (89% reduction)


## Key takeaways

| Concept | Statement |
|---|---|
| Sample size / MDE | Standard two-sample formulas; invert to ask "what can we detect with the traffic we have?" |
| CUPED | Reduces variance by $(1-\rho^2)$ using pre-period covariates — full derivation in experimental_design_course/08 |
| Skewed revenue metrics | Delta-method variance + heavy tails => much larger n than conversion metrics; winsorizing trades tail sensitivity for power |

## References

- Kohavi, R., Tang, D., & Xu, Y. (2020). *Trustworthy Online Controlled Experiments*, Ch. 4 & 19 (power, variance reduction).
- Deng, A., Xu, Y., Kohavi, R., & Walker, T. (2013). Improving the Sensitivity of Online Controlled Experiments by Utilizing Pre-Experiment Data. *WSDM 2013*.
- See `experimental_design_course/08_online_experiments.ipynb` for the full CUPED derivation and optimal-theta estimation.
